### **nef_translocation - PART 1 - From raw file to segmentation**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2025/10/14

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [ ]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
# from scipy.ndimage import median_filter, binary_fill_holes #to check
# from skimage.filters import threshold_otsu #to check
# from skimage.measure import label #to check
from skimage.transform import resize
from ome_types import to_xml
from utils.listdirNHF import listdirNHF
from utils.mksubdir import mkdir_tree
from image_preparation.extract_metadata import extract_bioio_scene_metadata
from image_preparation.name_metadata import extract_name_metadata
from image_preparation.make_imagej_metadata import imagej_compatible_metadata_dict
from utils.open_image import bioio_open_image
from utils.save_image import tifffile_save_ometiff
from image_preparation.save_metadata import save_xml_string
from image_preparation.preprocess_image import preprocess_fov, preprocess_channel



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modified are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the path to the directory storing the input images
input_directory = r"Z:\AlessandroUlivi_Data\projects\nef_translocation\data\15min"
# input_directory = r"Z:\AlessandroUlivi_Data\projects\nef_translocation\data\Nef-GFP 5min"
# input_directory = r"C:\Users\AG Fackler\Cristina\input_directory"

# indicate the path to the directory where outputs will be saved
output_directory = r"Z:\AlessandroUlivi_Data\projects\nef_translocation\develop\250916_pipeline_development"
# output_directory = r"C:\Users\AG Fackler\Cristina\output_directory"

# Indicate the markers used for each channel - if possible, try to be the most explicit possible (e.g. indicate dapi instead of nucleus,
# indicate phalloidin instead of cytoskeleton)
channel_0='ch638'
channel_1='actin'
channel_2='gfp'
channel_3='dapi'
channel_4='DIA'

# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

# inticate the unit of the x and y physical size of the images as extracted from the .nd2 raw file metadata
size_unit = 'micron' # should be implemented using image_preparation.extract_metadata.extract_physical_size_unit_from_nd2_xml()

# separator - used to split file .nd2 raw file name and extract metadata information within it
file_name_separator = '_'

# location
location="Center for Integrative Infectious Disease Research - Heidelberg (Germany)"

# microscope
microscope='Nikon Ti2 - CrestOptics X-Light V3 - spinning disc'

# objective
objective='Plan Apo Lambda D 100x Oil/1,45/0,13'

# ome suffix - used to save ome.tif files
ome_suffix = ".ome.tif"

# Indicate the part of the input file name to use to filter input files to analyse from files to ignore
# The input file contains .nd2 - input_file_exclude set to None indicates that no sub-string should be used to identify
# files in the input directory which should not be processed. As a consequence, all files containing the .nd2 string will be
# processed
input_file_target = ".nd2"
input_file_exclude = None

# indicate the name of the final order of the dimensions in the saved ome file
dims_order_name = 'dims_order'

# indicate the position of the axis containing the channels in the image
channel_axis=0

# nucleus position - fields of views (also called scenes) are individual acquisitions of different positions within samples.
# The pipeline expects 5 channels per field of view. At least one should be the nucleus, and at least one should be the f-actin
# staining. It is important to isolate these channels for the segmentation of cells and nuclei. The present parameters allow
# to specify the position of these channels along the channel axis of the field of view array in the scene.
# The position is 0-based, meaning that the first channel is at position 0, the second at position 1, etc.
nucleus_position = 3
actin_position = 1

# size of median filter applied to the image before segmentation a squared kernel of indicated sizes will be used
med_size_nucleus = 10
med_size_cell = 3

# the downsampling factor to apply to the images before passing them to cellpose. Factor 2 means that the image will be
# downsampled to half of its original size in all dimensions except the axis onte which nucleus and f-actin channels have
# been stacked
factor=2

# the order of the interpolation used to upsample the segmentation masks after cellpose segmentation. Refer to
# skimage.transform.resize documentation https://scikit-image.org/docs/0.25.x/api/skimage.transform.html#skimage.transform.resize
resize_order = 0

# Cell pose parameters
diameter = 150 # the avarage diameter of the cells in pixels. If set to None, cellpose will try to estimate it automatically
flow_threshold=0.4 # the threshold for the flow error. If the flow error is above this value, the cell will not be segmented
cellprob_threshold=0.0 # the threshold for the cell probability. If the cell probability is below this value, the cell will not be segmented

# # number of bins used to calculate the threshould for otsu segmentation of nucleus - this is not used at the moment
# otsu_nbins = 100 #to check

# indicate the datatype to give to the nucleus segmentation mask before converting it to a label image.
# The datatype should be able to allocate a number of values higher than the nuber of segmented cells in the image
# this is not used at the moment
# nucleus_label_img_dtype = np.uint16


# cell suffix - the suffix added to the cell segmentation masks
cell_suffix = "_Cl"

# nucleus suffix - the suffix added to the nucleus segmentation masks
nucleus_suffix = "_Nc"

# cytosol suffix - the suffix added to the cytosol segmentation masks - as the cytosol segmentation has been moved to part 2,
# this is not used at the moment
# cytosol_suffix = "_Ct" #to check



### Create output directory tree

Run the following cell.

Don't modify the following cell.

In [3]:
# Create output directory tree and get corresponding path objects - don't modify the following lines
secondary_output_path, metadata_directory, original_metadata_directory, proc_file_info_metadata_directory, fov_directory, segmentation_directory = mkdir_tree(secondary_output_name='secondary_output',
                                                                                                                                                      secondary_output_parent=os.getcwd(),
                                                                                                                                                      metadata_name='metadata',
                                                                                                                                                      metadata_parent=output_directory,
                                                                                                                                                      original_name='original',
                                                                                                                                                      proc_file_info_name='proc_file_info',
                                                                                                                                                      fov_name='fov',
                                                                                                                                                      fov_parent=output_directory,
                                                                                                                                                      segmentation_name='seg',
                                                                                                                                                      segmentation_parent=output_directory)


### Import the names of the raw files to process in a list

Run the following cell.

Don't modify the following cell.

In [4]:
# Import input file names as a list - don't modify the following line
input_file_list = listdirNHF(Path(input_directory),
                             target=input_file_target,
                             exclude=input_file_exclude)


### 1st MAIN LOOP
#### Extract field of views (also called scenes) to analyse from raw input files.
#### 1.1. Extract original metadata.
#### 1.1. Save original metadata.
#### 1.2. Extract metadata from file and scene names
#### 1.3. Save individual fields of view as ome.tif along with their metadata

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [ ]:

# intialize lists to collect all file metadata
excluded_scenes = []
scenes_metadata_collection = []

# iterate through the target files in the input directory
for input_file in input_file_list[:3]:
    
    print("=========")
    print((f"working on {input_file}"))

    # ---------   ---------
    # READ THE FILE AND THE ORIGINAL METADATA
    # ---------   ---------
    bioio_input_image, input_image_metadata = bioio_open_image(os.path.join(input_directory, input_file),
                                                            return_metadata=True)

    # ---------   ---------
    # SAVE THE ORIGINAL METADATA
    # ---------   ---------

    # convert input_image_metadata to xml object
    xml_input_image_metadata = to_xml(input_image_metadata)


    # save file metadata in secondary output
    save_xml_string(xml_input_image_metadata,
                    os.path.join(original_metadata_directory,
                                f"{input_file.removesuffix(input_file_target)}_string.xml")
                                )
    

    # ---------   ---------
    # ITERATE THROUGH THE SCENES WITHIN THE FILE
    # ---------   ---------
    # iterate through the scenes
    for scene_n, scene in enumerate(bioio_input_image.scenes):
        
        print("---------")
        print(f"working on scene {scene}")

        # set scene
        bioio_input_image.set_scene(scene)

        # ---------   ---------
        # EXTRACT METADATA FROM FILE NAME
        # ---------   ---------

        # extract metadata from file name
        name_metadata_dict = extract_name_metadata(input_file,
                                                   separator=file_name_separator,
                                                   infobits={'donor':(0,None,None,None),
                                                             'processing_date':datetime.datetime.now().strftime('%y%m%d'),
                                                             'transfection':(1,None,None,None),
                                                             'stiffness':(2,None,None,None),
                                                             'stimulation':(3,None,None,None),
                                                             'time_of_stimulation':(4,None,None,None)})
        
                        
        # form the saving name of the file
        save_file_name = f"{input_file.removesuffix(input_file_target)}{file_name_separator}s{scene_n}{ome_suffix}"

        # extract metadata from raw image metadata, add extra info, return metadata in their final version
        scene_metadata_series, scene_metadata_dict = extract_bioio_scene_metadata(bioio_scene=bioio_input_image,
                                                                                  dims_order_name=dims_order_name,
                                                                                  raw_file_name=input_file,
                                                                                  scene_name=scene,
                                                                                  processing_date_yymmdd=name_metadata_dict['processing_date'],
                                                                                  ome_tif_file_name=save_file_name,
                                                                                  location=location,
                                                                                  microscope=microscope,
                                                                                  objective=objective,
                                                                                  donor=name_metadata_dict['donor'],
                                                                                  transfection=name_metadata_dict['transfection'],
                                                                                  stiffness=name_metadata_dict['stiffness'],
                                                                                  stimulation=name_metadata_dict['stimulation'],
                                                                                  time_of_stimulation=name_metadata_dict['time_of_stimulation'],
                                                                                  channel_0=channel_0,
                                                                                  channel_1=channel_1,
                                                                                  channel_2=channel_2,
                                                                                  channel_3=channel_3,
                                                                                  channel_4=channel_4,
                                                                                  physical_size_unit_x=size_unit,
                                                                                  physical_size_unit_y=size_unit)
        
        # ---------   ---------
        # COLLECT SCENE METADATA IN THE COLLECTION LIST
        # ---------   ---------

        # append scene_metadata_series to collection list
        scenes_metadata_collection.append(scene_metadata_series)

        # ---------   ---------
        # SAVE OME.TIF FILE
        # ---------   ---------
        # get data as an array
        input_scene = bioio_input_image.data

        # Remove axis of size 1 - this will make the image compatible with ImageJ
        input_scene = np.squeeze(input_scene)

        # change metadata dictionary to an ImageJ comaptible format
        imagej_scene_metadata_dict = imagej_compatible_metadata_dict(scene_metadata_dict)

        tifffile_save_ometiff(os.path.join(fov_directory,save_file_name),
                              data=input_scene,
                              imagej=True,
                              photometric="minisblack",
                              metadata=imagej_scene_metadata_dict)

# ---------   ---------
# FORM A DATA FRAME WITH ALL METADATA
# ---------   ---------
# concatenate scenes metadata into a pandas data frame
metadata_df = pd.concat(scenes_metadata_collection, axis=1).T

print("")
print("Finished")


working on D26_GFP_Glass_CD3CD28_15min_.nd2
---------
working on scene A1
---------
working on scene A2
---------
working on scene A3
---------
working on scene B3
---------
working on scene B2
---------
working on scene B1
---------
working on scene C1
---------
working on scene C2
---------
working on scene C3
working on D26_GFP_Glass_CD3CD28_15min_001.nd2
---------
working on scene A1
---------
working on scene A2
---------
working on scene A3
---------
working on scene B3
---------
working on scene B2
---------
working on scene B1
---------
working on scene C1
---------
working on scene C2
---------
working on scene C3
working on D26_GFP_Glass_CD3CD28_15min_002.nd2
---------
working on scene A1
---------
working on scene A2
---------
working on scene A3
---------
working on scene B3
---------
working on scene B2
---------
working on scene B1
---------
working on scene C1
---------
working on scene C2
---------
working on scene C3

Finished


# Segmentation of cell, nucleus and cytosol

### Import CellPose and CellPose models - NOTE: this is kept separated from previous imports to allow notebook modularity

Run the following cell.

Don't modify the following cell.

In [8]:
# Import required modules
from cellpose import models
from cellpose.io import imread
import torch

# Use gpu if available else cpu
if torch.cuda.is_available():
    use_gpu = True
    print("---------")
    print("GPU is available")
else:
    use_gpu = False
    print("---------")
    print("GPU is not available")

# Import segmentation model
model = models.CellposeModel(gpu=use_gpu)




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.12.11 
torch version:  	2.5.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


---------
GPU is not available


### 2nd MAIN LOOP
#### 2.1. Preprocess field of views.
#### 2.2. Segment cell and nucleus.
#### 2.2. Save segmentation masks
#### 2.3. Update and save metadata

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [ ]:

seg_mentation_metadata_dict = {f'segmentation_date_yymmdd{nucleus_suffix}{cell_suffix}':[], #to check
                             'cell_segmentation':[], #to check
                             'nucleus_segmentation':[]} #to check

# iterate through the rows of the metadata dataframe
for file_idx in metadata_df.index[:3]:
    print("---------")

    # get the name of the field of view as ome.tif file
    fov_ome_name = metadata_df.loc[file_idx, 'ome_tif_file_name']

    #read file if it was processed
    if fov_ome_name not in [np.nan]:
        print(f"working on {fov_ome_name}")
        
        # ---------   ---------
        # OPEN THE FIELD OF VIEW FILE
        # ---------   ---------
        img = imread(os.path.join(fov_directory,fov_ome_name))

        # ---------   ---------
        # PREPROCESS IMAGE
        # Select nucleus and actin channels
        # filter nucleus using a 10x10 median filtering
        # stack non-filtered nucleus and actin-channel into a new array and filter the channels, individually, using a 3x3 median filtering
        # 2x2 binning
        # NOTE: no dtype conversion is needed as resizing automatically changes image to float
        # ---------   ---------

        # Select nucleus and actin channels
        unstacked_img = np.unstack(img, axis=channel_axis)
        nucleus_channel = unstacked_img[nucleus_position]
        actin_channel = unstacked_img[actin_position]
        restacked_img = np.stack([nucleus_channel, actin_channel], axis=channel_axis)

        # Preprocess nucleus and nucleus-actin-stack
        preproc_nucleus = preprocess_channel(image=nucleus_channel,
                                             med_size=med_size_nucleus,
                                             factor=factor)

        preproc_img = preprocess_fov(image=actin_channel,
                                       channel_axis=channel_axis,
                                       med_size=med_size_cell,
                                       factor=factor)
        
        print("Image preprocessing is done")

        # ---------   ---------
        # SEGMENT CELLS
        # ---------   ---------

        # calculate the original size of the 2D image (aka, the size of each imaged channel)
        # this will be used to resize the segmented masks to their original size, as masks are calculated on binned images
        original_ch_img_size = tuple([s for p,s in enumerate(img.shape) if p!=channel_axis])

        print("Cellular segmentation is beginning. Please wait...")
        cell_masks_i, cell_flows, cell_styles = model.eval(preproc_img, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold, diameter=diameter)
        
        # re-expand segmentation mask
        cell_masks = resize(cell_masks_i, output_shape=original_ch_img_size, order=resize_order)

        print("Cellular segmentation is done")

        # ---------   ---------
        # SEGMENT NUCLEI
        # NOTE: no filter diameter is used, small structures are useful to be detected, so that they are subtracted from cytosol masks
        # ---------   ---------
        print("Nuclear segmentation is beginning. Please wait...")
        nuc_masks_i, nuc_flows, nuc_styles = model.eval(preproc_nucleus, flow_threshold=0.9, cellprob_threshold=0.5) #to check
        
        # re-expand segmentation mask
        nuc_masks = resize(nuc_masks_i, output_shape=original_ch_img_size, order=resize_order) #to check
        
        print("Nuclear segmentation is done")

        # # ---------   ---------
        # # SEGMENT CYTOSOLS - THIS HAS BEEN MOVED TO PART 2
        # # ---------   ---------
        # print("Cytoplasm segmentation is beginning. Please wait...") #to check
        # cyt_masks = np.where(nuc_masks>0,0,cell_masks).astype(np.uint16) #to check
        # print("Cytoplasm segmentation is done") #to check

        # ---------   ---------
        # SAVE RESULTS
        # ---------   ---------
        
        # form saving names
        cell_save_name = f"{fov_ome_name.removesuffix(ome_suffix)}{cell_suffix}{ome_suffix}"

        nuc_save_name = f"{fov_ome_name.removesuffix(ome_suffix)}{nucleus_suffix}{ome_suffix}"
        
        # cyt_save_name = f"{fov_ome_name.removesuffix(ome_suffix)}{cytosol_suffix}{ome_suffix}" #to check

        # form a metadata dictionary to be used for saving metadata in the segmentation masks
        segmentation_metadata_dict = {'processing_date_yymmdd':datetime.datetime.now().strftime('%y%m%d')}
        for clm in metadata_df.columns:
            if clm in ['raw_file_name', 'scene_name', 'ome_tif_file_name', 'location', 'microscope', 'objective', 'donor',
                       'transfection', 'stiffness', 'stimulation', 'time_of_stimulation',
                       'physical_size_unit_x', 'physical_size_unit_y', 'physical_size_y', 'size_x',
                       'physical_size_x']:
                
                segmentation_metadata_dict[clm]=metadata_df.loc[file_idx,clm]
        
        # change metadata dictionary to an ImageJ comaptible format
        imagej_segmentation_metadata_dict = imagej_compatible_metadata_dict(segmentation_metadata_dict)

        # save file
        tifffile_save_ometiff(os.path.join(segmentation_directory,cell_save_name),
                                    data=cell_masks,
                                    imagej=True,
                                    photometric="minisblack",
                                    metadata=imagej_segmentation_metadata_dict)

        tifffile_save_ometiff(os.path.join(segmentation_directory,nuc_save_name),
                                    data=nuc_masks,
                                    imagej=True,
                                    photometric="minisblack",
                                    metadata=imagej_segmentation_metadata_dict)
        
        # tifffile_save_ometiff(os.path.join(segmentation_directory,cyt_save_name), #to check
        #                             data=cyt_masks, #to check
        #                             imagej=True, #to check
        #                             photometric="minisblack", #to check
        #                             metadata=imagej_segmentation_metadata_dict) #to check

        print("Segmentation results have been saved")

        # ---------   ---------
        # COLLECT FILE SAVE NAME AND SEGMENTATION DATE TO UPDATE METADATA DATAFRAME
        # ---------   ---------
        # seg_mentation_metadata_dict['segmentation_date_yymmdd'].append(datetime.datetime.now().strftime('%y%m%d')) #to check
        seg_mentation_metadata_dict[f'segmentation_date_yymmdd{nucleus_suffix}{cell_suffix}'].append(datetime.datetime.now().strftime('%y%m%d')) #to check
        seg_mentation_metadata_dict['cell_segmentation'].append(cell_save_name)
        seg_mentation_metadata_dict['nucleus_segmentation'].append(nuc_save_name)
        # seg_mentation_metadata_dict['cytosol_segmentation'].append(cyt_save_name) #to check

    # update metadata_update_dict with nan values if the file was not processed
    else:
        # seg_mentation_metadata_dict['segmentation_date_yymmdd'].append(np.nan) #to check
        seg_mentation_metadata_dict[f'segmentation_date_yymmdd{nucleus_suffix}{cell_suffix}'].append(np.nan) #to check
        seg_mentation_metadata_dict['cell_segmentation'].append(np.nan)
        seg_mentation_metadata_dict['nucleus_segmentation'].append(np.nan)
        # seg_mentation_metadata_dict['cytosol_segmentation'].append(np.nan) #to check

print("")
print("Segmentation completed")

# ---------   ---------
# UDDATE METADATA DICTIONARY
# ---------   ---------

# use seg_mentation_metadata_dict to for a dataframe
segmentation_metadata_df = pd.DataFrame.from_dict(seg_mentation_metadata_dict)

# add segmentation information to the metadata dataframe
glob_metadata_df = pd.concat([metadata_df,segmentation_metadata_df], axis=1)


# ---------   ---------
# SAVE THE FINAL METADATA FILE
# ---------   ---------
glob_metadata_df.to_csv(os.path.join(proc_file_info_metadata_directory, f"{datetime.datetime.now().strftime('%y%m%d')}_nef_translocation_metadata.csv"))

print("")
print("Processing metadata saved")



---------
working on D26_GFP_Glass_CD3CD28_15min__s0.ome.tif
uint16
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
(2720, 1360)
Cellular segmentation is done
Nuclear segmentation is beginning. Please wait...
Nuclear segmentation is done
Segmentation results have been saved
---------
working on D26_GFP_Glass_CD3CD28_15min__s1.ome.tif
uint16
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
(2720, 1360)
Cellular segmentation is done
Nuclear segmentation is beginning. Please wait...
Nuclear segmentation is done
Segmentation results have been saved
---------
working on D26_GFP_Glass_CD3CD28_15min__s2.ome.tif
uint16
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
(2720, 1360)
Cellular segmentation is done
Nuclear segmentation is beginning. Please wait...
Nuclear segmentation is done
Segmentation results have been saved

Segmentation completed

Processing metadata saved


### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# collect hyperparameters in a dictionary

hyperparameter_dict = {'input_directory':input_directory,
                       'output_directory':output_directory,
                       'channel_0':channel_0,
                       'channel_1':channel_1,
                       'channel_2':channel_2,
                       'channel_3':channel_3,
                       'channel_4':channel_4,
                       'size_unit':size_unit,
                       'file_name_separator':file_name_separator,
                       'location':location,
                       'microscope':microscope,
                       'objective':objective,
                       'nucleus_position':nucleus_position,
                       'actin_position':actin_position,
                       'input_file_target':input_file_target,
                       'input_file_exclude':input_file_exclude,
                       'ome_suffix':ome_suffix,
                       'med_size_nucleus':med_size_nucleus,
                       'med_size_cell':med_size_cell,
                       'channel_axis':channel_axis,
                       'dims_order_name':dims_order_name,
                       'factor':factor,
                       'use_gpu':use_gpu,
                       'diameter':diameter,
                       'flow_threshold':flow_threshold,
                       'cellprob_threshold':cellprob_threshold,
                       'resize_order':resize_order,
                       'cell_suffix':cell_suffix,
                       'nucleus_suffix':nucleus_suffix}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}_nef_translocation_hyperparameters_part1.csv"
hyperparameter_series.to_csv(os.path.join(secondary_output_path,hyperparameter_saving_name))
